<a href="https://colab.research.google.com/github/aaricaaric018-del/smart-machine-health-monitoring/blob/main/Smart_Machine_Health_Monitoring_AI_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
import joblib

df = pd.read_csv("ai4i2020.csv")
df.head()

df.info()

df.isnull().sum()

df.describe()

df_clean = df.copy()

df_clean = df_clean.drop(columns=[
    "UDI",
    "Product ID",
    "TWF",
    "HDF",
    "PWF",
    "OSF",
    "RNF"
])

df_clean.head()

le = LabelEncoder()

df_clean["Type"] = le.fit_transform(df_clean["Type"])

print(le.classes_)

df_clean.head()

df_clean["Machine failure"].value_counts()
plt.figure(figsize=(6,4))

sns.countplot(
    x="Machine failure",
    data=df_clean
)

plt.title("Machine Failure Distribution")
plt.show()
plt.figure(figsize=(10,8))

sns.heatmap(
    df_clean.corr(),
    annot=True,
    cmap="coolwarm"
)

plt.title("Correlation Heatmap")
plt.show()
df_clean.hist(figsize=(12,10))
plt.show()
plt.figure(figsize=(8,5))

sns.boxplot(x=df_clean["Torque [Nm]"])

plt.title("Torque Distribution")
plt.show()



In [21]:
X = df_clean.drop("Machine failure", axis=1)
y = df_clean["Machine failure"]
X.head()
y.head()
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(8000, 6)
(2000, 6)
(8000,)
(2000,)


In [ ]:
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression(max_iter=1000)

model_lr.fit(X_train, y_train)

y_pred_lr = model_lr.predict(X_test)
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

accuracy = accuracy_score(y_test, y_pred_lr)

print("Accuracy:", accuracy)
cm = confusion_matrix(y_test, y_pred_lr)

print(cm)

print(classification_report(y_test, y_pred_lr))
plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Logistic Regression - Confusion Matrix")

plt.show()

In [ ]:
from sklearn.tree import DecisionTreeClassifier
model_dt = DecisionTreeClassifier(
    random_state=42
)
model_dt.fit(X_train, y_train)
y_pred_dt = model_dt.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred_dt))
cm_dt = confusion_matrix(y_test, y_pred_dt)

print(cm_dt)
print(classification_report(y_test, y_pred_dt))
plt.figure(figsize=(6,5))

sns.heatmap(
    cm_dt,
    annot=True,
    fmt="d",
    cmap="Greens"
)

plt.title("Decision Tree Confusion Matrix")

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
model_rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
model_rf.fit(X_train, y_train)
y_pred_rf = model_rf.predict(X_test)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
cm_rf = confusion_matrix(y_test, y_pred_rf)

print(cm_rf)
print(classification_report(y_test, y_pred_rf))
plt.figure(figsize=(6,5))

sns.heatmap(
    cm_rf,
    annot=True,
    fmt="d",
    cmap="Oranges"
)

plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
# Import required libraries
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# Rename columns to remove special characters
df_clean.columns = [
    "Type",
    "Air_temperature",
    "Process_temperature",
    "Rotational_speed",
    "Torque",
    "Tool_wear",
    "Machine_failure"
]

# Separate features and target
X = df_clean.drop("Machine_failure", axis=1)
y = df_clean["Machine_failure"]

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Create the XGBoost model
model_xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    eval_metric="logloss"
)

# Train the model
model_xgb.fit(X_train, y_train)

# Make predictions
y_pred_xgb = model_xgb.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred_xgb)
print("Accuracy:", accuracy)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_xgb)
print("\nConfusion Matrix:")
print(cm)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

# Plot Confusion Matrix
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples")
plt.title("XGBoost Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [33]:
import joblib

# Save the trained XGBoost model
joblib.dump(model_xgb, "xgboost_model.pkl")

# Save the Label Encoder
joblib.dump(le, "label_encoder.pkl")

print("Model saved successfully!")


Model saved successfully!


In [ ]:
# Load the model
loaded_model = joblib.load("xgboost_model.pkl")

# Load the encoder
loaded_encoder = joblib.load("label_encoder.pkl")

print("Model loaded successfully!")
sample = [[
    1,        # Type (encoded)
    298.1,    # Air_temperature
    308.6,    # Process_temperature
    1551,     # Rotational_speed
    42.8,     # Torque
    0         # Tool_wear
]]

prediction = loaded_model.predict(sample)

print(prediction)

In [ ]:
from xgboost import plot_importance
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

plot_importance(
    model_xgb,
    max_num_features=6,
    importance_type='gain'
)

plt.title("Feature Importance - XGBoost")
plt.show()

In [36]:
sample = pd.DataFrame({
    "Type": [1],
    "Air_temperature": [298.1],
    "Process_temperature": [308.6],
    "Rotational_speed": [1551],
    "Torque": [42.8],
    "Tool_wear": [0]
})

prediction = model_xgb.predict(sample)

if prediction[0] == 0:
    print("✅ Machine is Healthy")
else:
    print("⚠️ Machine Failure Predicted")

✅ Machine is Healthy


In [ ]:
probability = model_xgb.predict_proba(sample)

print(probability)

In [ ]:
def predict_machine_health(
    machine_type,
    air_temp,
    process_temp,
    rpm,
    torque,
    tool_wear
):
    sample = pd.DataFrame({
        "Type": [machine_type],
        "Air_temperature": [air_temp],
        "Process_temperature": [process_temp],
        "Rotational_speed": [rpm],
        "Torque": [torque],
        "Tool_wear": [tool_wear]
    })

    prediction = model_xgb.predict(sample)[0]
    probability = model_xgb.predict_proba(sample)[0][1]

    return prediction, probability
    prediction, probability = predict_machine_health(
    1,
    298.1,
    308.6,
    1551,
    42.8,
    0
)

print(prediction)
print(probability)

In [ ]:
prediction = model_xgb.predict(sample)[0]
probability = model_xgb.predict_proba(sample)[0]

failure_probability = probability[1] * 100
healthy_probability = probability[0] * 100

print("=" * 40)
print("Machine Health Prediction")
print("=" * 40)

if prediction == 0:
    print("Status : HEALTHY")
else:
    print("Status : FAILURE DETECTED")

print(f"Healthy Probability : {healthy_probability:.2f}%")
print(f"Failure Probability : {failure_probability:.2f}%")


In [42]:
def predict_machine_health(
    machine_type,
    air_temp,
    process_temp,
    rpm,
    torque,
    tool_wear
):
    sample = pd.DataFrame({
        "Type": [machine_type],
        "Air_temperature": [air_temp],
        "Process_temperature": [process_temp],
        "Rotational_speed": [rpm],
        "Torque": [torque],
        "Tool_wear": [tool_wear]
    })

    prediction = model_xgb.predict(sample)[0]
    probability = model_xgb.predict_proba(sample)[0]

    healthy_prob = probability[0] * 100
    failure_prob = probability[1] * 100

    print("=" * 50)
    print("SMART MACHINE HEALTH MONITOR")
    print("=" * 50)

    if prediction == 0:
        print("Machine Status : HEALTHY")
    else:
        print("Machine Status : FAILURE PREDICTED")

    print(f"Healthy Probability : {healthy_prob:.2f}%")
    print(f"Failure Probability : {failure_prob:.2f}%")

    return prediction
    predict_machine_health(
    machine_type=1,
    air_temp=298.1,
    process_temp=308.6,
    rpm=1551,
    torque=42.8,
    tool_wear=0
)

In [46]:
import joblib

joblib.dump(model_xgb, "xgboost_model.pkl")
joblib.dump(le, "label_encoder.pkl")
from google.colab import files

files.download("xgboost_model.pkl")
files.download("label_encoder.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from xgboost import plot_importance
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

plot_importance(
    model_xgb,
    importance_type="gain",
    max_num_features=6,
    height=0.8
)

plt.title("Feature Importance using XGBoost")
plt.show()

In [ ]:
import pandas as pd

importance = model_xgb.feature_importances_

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importance
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance)

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# Predict probabilities for the positive class
y_prob = model_xgb.predict_proba(X_test)[:, 1]

# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

# Calculate AUC
auc_score = roc_auc_score(y_test, y_prob)

print("AUC Score:", round(auc_score, 4))

# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f"XGBoost (AUC = {auc_score:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="red")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - XGBoost")
plt.legend()
plt.grid(True)

plt.show()

# Smart Machine Health Monitoring & Predictive Maintenance

## AI/ML Module

### Team Member:
Bhavithiran S

### Role:
AI/ML Engineer

### Objective
Develop a machine learning model that predicts machine failures using sensor data to enable predictive maintenance and reduce unexpected downtime.

### Dataset
AI4I 2020 Predictive Maintenance Dataset

### Algorithms Used
- Logistic Regression
- Decision Tree
- Random Forest
- XGBoost

### Final Selected Model
XGBoost

## 1. Import Required Libraries
## 2. Load the Dataset

## 3. Data Understanding

## 4. Data Preprocessing
df_clean
## 5. Exploratory Data Analysis
## 6. Feature Selection and Data Splitting
## 7. Machine Learning Models
## 8. Model Comparison
## 9. Feature Importance
### Observation

Torque, Rotational Speed, and Tool Wear are the three most influential features for predicting machine failures.
## 10. ROC Curve and AUC Score
### Result

The XGBoost model achieved an AUC score of **0.9754**, indicating excellent discrimination between healthy and failed machines.
## 11. Save the Trained Model
## 12. Testing with New Machine Data
# Conclusion

Four machine learning algorithms were developed and evaluated for predictive maintenance.

Among them, XGBoost achieved the best overall performance.

Final Results:

• Accuracy : 98.55%

• Precision : 85%

• Recall : 64%

• F1 Score : 73%

• ROC-AUC : 97.54%

The trained model can be integrated into a web-based monitoring dashboard to predict machine failures in real time and support preventive maintenance decisions.

In [53]:
comparison = pd.DataFrame({
    "Model":[
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy":[
        0.9725,
        0.9780,
        0.9840,
        0.9855
    ],
    "Precision":[
        0.62,
        0.63,
        0.84,
        0.85
    ],
    "Recall":[
        0.26,
        0.69,
        0.59,
        0.64
    ],
    "F1 Score":[
        0.37,
        0.66,
        0.69,
        0.73
    ]
})

comparison

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.9725,0.62,0.26,0.37
1,Decision Tree,0.9780,0.63,0.69,0.66
2,Random Forest,0.9840,0.84,0.59,0.69
3,XGBoost,0.9855,0.85,0.64,0.73
